# 最大割问题

**类别：** 选址

来源：[https://www.hexaly.com/templates/max-cut-problem](https://www.hexaly.com/templates/max-cut-problem)


## 问题

**在最大割问题**中，我们考虑一个图 G = (V, E)。我们希望找到该图的一个最大割，即将图的顶点划分为两个互补集合 S 和 T，使得 S 和 T 之间的边数尽可能大。等价地，该问题在于找到该图的一个尽可能多边的二部子图。这里，我们考虑该问题的一个更通用的版本：加权最大割问题。每条边都与一个数（其权重）相关联，问题的目标是找到一个顶点的子集 S，使得 S 与其补集之间的边权重之和尽可能大。

	

### 学到的建模原则

- 了解 Hexaly Optimizer 的建模风格：[区分决策变量与中间表达式](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variables)
- 使用 [`非线性算子`](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#table-of-available-operators-and-functions) 来确定图中每条边是否在割集中


## 数据

我们提供的最大割问题实例来自 [Biq Mac Library](http://biqmac.uni-klu.ac.at/biqmaclib.html)。最优解和每个数据集的描述可在此处找到 [此处](http://biqmac.uni-klu.ac.at/biqmaclib.pdf)。数据文件的格式如下：

- 顶点数
- 边数
- 带边权重的邻接表


## 模型

最大割问题的Hexaly模型使用 布尔决策变量 表示每条边是否属于子集 S。

由其起点和终点顶点描述的一条边位于割集中，当且仅当恰好有一个顶点属于 S。使用非线性 **neq** 算子，我们确定每条边是否在割集中。然后我们可以计算目标函数的值，即割集中所有边的权重之和。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]

#
# Read instance data
#
def read_instance(filename):
    file_it = iter(read_integers(filename))
    # Number of vertices
    n = next(file_it)
    # Number of edges
    m = next(file_it)

    # Origin of each edge
    origin = [None] * m
    # Destination of each edge
    dest = [None] * m
    # Weight of each edge
    w = [None] * m

    for e in range(m):
        origin[e] = next(file_it)
        dest[e] = next(file_it)
        w[e] = next(file_it)
    
    return n, m, origin, dest, w

def main(instance_file, output_file, time_limit):
    n, m, origin, dest, w = read_instance(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Decision variables x[i]
        # True if vertex x[i] is on the right side of the cut
        # and false if it is on the left side of the cut
        x = [model.bool() for i in range(n)]

        # An edge is in the cut-set if it has an extremity in each class of the bipartition
        incut = [None] * m
        for e in range(m):
            incut[e] = model.neq(x[origin[e] - 1], x[dest[e] - 1])

        # Size of the cut
        cut_weight = model.sum(w[e] * incut[e] for e in range(m))
        model.maximize(cut_weight)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        #  - objective value
        #  - each line contains a vertex number and its subset (1 for S, 0 for V-S)
        #
        if output_file != None:
            with open(output_file, 'w') as f:
                f.write("%d\n" % cut_weight.value)
                # Note: in the instances the indices start at 1
                for i in range(n):
                    f.write("%d %d\n" % (i + 1, x[i].value))

if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python maxcut.py inputFile [outputFile] [timeLimit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 10
    main(instance_file, output_file, time_limit)
